# Intraday volume periodicity — illustration pedagogique CPU-only

## Origine et portee

Ce notebook illustre le concept spectral derriere l'article QuantConnect *Intraday Volume Periodicity* (#21066) sans pretendre implementer le rotator decrit par l'auteur. Il prend pour reference la source primaire du phenomene :

> Wu, L., Zhang, R. & Dai, Y. *Spectral Volume Models: Universal High-Frequency Periodicities in Intraday Trading Activities*. **Management Science** (nov. 2025), doi:10.1287/mnsc.2024.06215 ; preprint SSRN 4230610.

La these des auteurs est que le trading algorithmique produit des **periodicites persistantes** dans le flux de volume intraday, et que ces periodicites portent une prime de selection adverse exploitable cross-section. Ce notebook ne porte aucun claim de profitabilite : il illustre la **detection spectrale** sur une serie synthetique ou la periodicite est connue par construction, pour montrer ce que ressortirait d'un signal periodique faible dans une serie reelle.

**Pourquoi CPU-only**. Pas de backtest QC Cloud ici : la machine d'execution n'a pas les identifiants QC_API_USER_ID et la livraison est pedagogique, pas un rotator de production. Voir `qc-cycle-gating-recoverable-user-hand` MEMORY pour le mecanisme de routage vers une lane QC.

**Verdict de l'issue #14991**. Suivant le triage de po-2024 (body de l'issue, acceptance deja cochee), le rotator lui-meme ne justifie ni projet ni notebook (signal proprietaire QC, in-sample, draft pending review, optimum au bord de grille). Ce notebook-ci est strictement l'illustration du concept spectral en amont : pedagogique, multi-cycles, sans pretention de rotator.

In [1]:
import numpy as np

rng = np.random.default_rng(seed=42)

# Fenetre U-shape typique d'une journee de trading US : pic a l'ouverture,
# creux le matin, pic au close, bruit multiplicatif. On y injecte une
# periodicite artificielle a 30 min (fond de panier du mid-day).
n_min = 390  # 6h30 = 390 minutes de trading US
t = np.arange(n_min)

u_shape = 1.0 + 1.2 * np.exp(-((t - 30) ** 2) / (2 * 25 ** 2)) \
              + 0.9 * np.exp(-((t - 380) ** 2) / (2 * 30 ** 2)) \
              + 0.3 * np.exp(-((t - 200) ** 2) / (2 * 60 ** 2))

periodicite = 0.25 * np.sin(2 * np.pi * t / 30)  # 30 minutes = 2 cycles/h
bruit = rng.lognormal(mean=0.0, sigma=0.6, size=n_min)
volume = u_shape * (1.0 + periodicite) * bruit

print(f"Serie synthetique generee : n={n_min} minutes, pic={volume.max():.1f},")
print(f"creux={volume.min():.1f}, SNR periodicite/bruit ~0.25/{np.std(bruit):.2f}")

Serie synthetique generee : n=390 minutes, pic=8.7,
creux=0.2, SNR periodicite/bruit ~0.25/0.76


In [2]:
# Spectre de puissance (FFT). On attend un pic a f = 1/30 min^-1 = 0.033 cycle/min,
# soit une periode de 30 minutes. Le U-shape produit aussi du contenu basse
# frequence, distinct du pic periodique.
from numpy.fft import rfft, rfftfreq

spectrum = np.abs(rfft(volume - volume.mean())) ** 2
freqs = rfftfreq(n_min, d=1.0)  # 1 pas = 1 minute

idx_top = np.argsort(spectrum)[::-1][:5]
print("Top 5 frequences detectees :")
for i in idx_top:
    f = freqs[i]
    period_min = 1.0 / f if f > 0 else float('inf')
    print(f"  f = {f:.4f} cycle/min  ->  periode {period_min:.1f} min  "
          f"(puissance {spectrum[i]:.0f})")

Top 5 frequences detectees :
  f = 0.0333 cycle/min  ->  periode 30.0 min  (puissance 10234)
  f = 0.0051 cycle/min  ->  periode 195.0 min  (puissance 5542)
  f = 0.0026 cycle/min  ->  periode 390.0 min  (puissance 4571)
  f = 0.0641 cycle/min  ->  periode 15.6 min  (puissance 2340)
  f = 0.1821 cycle/min  ->  periode 5.5 min  (puissance 2046)


## Lecture du resultat

Sur la serie synthetique, le pic de puissance le plus eleve (apres le continu DC) doit tomber sur la periode injectee de 30 minutes. Cela valide la methode : une FFT directe sur le volume intraday detecte une periodicite connue. Sur une serie reelle, la meme methode donne un pic dominant a la periodicite structurelle du marche (cf. Wu et al. 2025 — periodicites U-shape a 30 min et 60 min selon univers).

## Limites pedagogiques

Ce notebook n'aborde pas : (a) la stationnarite intraday (le U-shape produit du contenu basse frequence parasite), (b) le fenetrage de la FFT (Hann/Hamming), (c) la robustesse multi-jour (le spectre agrege), (d) la prime cross-section (selection adverse sur periodicite relative). Ce sont des grains distincts, hors du perimetre d'une illustration 3-cellules CPU-only.

L'article QC (#21066) et le papier M&S sous-jacent restent les references pour quiconque veut poursuivre ; ce notebook ne les remplace pas, il en montre un echo minimal.